# Tutorial: ColBERT vs. Cross-Encoder vs. MICE-P

This notebook is a practical companion to [MICE to Meet You](https://arxiv.org/abs/2610.31932). We will run three real public checkpoints over the same tiny corpus and inspect what each architecture computes.

The objective is architectural understanding, not a publishable benchmark. By the end, you should be able to explain **what is precomputed**, **where query–document interaction happens**, and **why retrieval speed and ranking quality trade off**.

## 0. Mental model

**Cross-encoder**

`[query tokens ; document tokens] → all transformer layers jointly → one relevance score`

Nothing document-specific can be fully cached before the query arrives. This is expressive, but expensive over a large corpus.

**ColBERT**

`query → query-token vectors` and `document → document-token vectors` independently, followed by `MaxSim`

Document token vectors can be precomputed and indexed. For each query token, MaxSim finds the most similar document token, then sums those best matches.

**MICE-P**

`document → lower contextualization layers → 128-d stored token states`

At query time, stored states are restored to the model dimension and passed through learned interaction layers with the query. This keeps richer learned interactions than MaxSim while avoiding repeated lower-layer document computation.

In [ ]:
%pip -q install -U "sentence-transformers>=6.0.0" "transformers>=5.8.1"

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer
from sentence_transformers import CrossEncoder, MultiVectorEncoder

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 1. One corpus, three queries, known answers

The corpus deliberately contains lexical traps: Jupiter has a *red spot* but is not the *Red Planet*; Jaguar can mean an animal or a car company. This makes the ranking behavior easier to discuss than a random benchmark sample.

In [ ]:
corpus = [
    {'id': 'd0', 'text': 'Mars is called the Red Planet because iron minerals in its soil oxidize and appear reddish.'},
    {'id': 'd1', 'text': 'Jupiter is the largest planet in the Solar System and has a famous Great Red Spot.'},
    {'id': 'd2', 'text': 'Venus is similar in size to Earth and is sometimes called Earth’s sister planet.'},
    {'id': 'd3', 'text': 'The jaguar is a large wild cat native to the Americas and often lives near forests and wetlands.'},
    {'id': 'd4', 'text': 'Jaguar produces luxury cars, including electric and sport utility vehicles.'},
    {'id': 'd5', 'text': 'Photosynthesis lets plants convert light energy into chemical energy.'},
    {'id': 'd6', 'text': 'Paris is the capital of France and lies along the Seine river.'},
    {'id': 'd7', 'text': 'Vitamin C contributes to normal immune function and collagen formation.'},
]

queries = [
    {'id': 'q0', 'text': 'Which planet is known as the Red Planet?', 'relevant': 'd0'},
    {'id': 'q1', 'text': 'Where does the jaguar animal naturally live?', 'relevant': 'd3'},
    {'id': 'q2', 'text': 'What is the capital city of France?', 'relevant': 'd6'},
]

doc_ids = [row['id'] for row in corpus]
documents = [row['text'] for row in corpus]
query_texts = [row['text'] for row in queries]
pd.DataFrame(corpus)

In [ ]:
def ranking_rows(model_name, scores):
    rows = []
    for qi, query in enumerate(queries):
        order = np.argsort(-scores[qi])
        for rank, di in enumerate(order, start=1):
            rows.append({
                'model': model_name,
                'query_id': query['id'],
                'rank': rank,
                'doc_id': doc_ids[di],
                'relevant': doc_ids[di] == query['relevant'],
                'score': float(scores[qi, di]),
            })
    return rows

def reciprocal_ranks(scores):
    values = []
    for qi, query in enumerate(queries):
        ranked_ids = [doc_ids[i] for i in np.argsort(-scores[qi])]
        values.append(1.0 / (ranked_ids.index(query['relevant']) + 1))
    return values

## 2. Cross-encoder: full joint attention

A cross-encoder sees the query and one document in the same input sequence. Every transformer layer can let a query token attend to document tokens and vice versa. The cost is one full model evaluation per query–document pair.

We use the 68M Ettin reranker, the cross-encoder family used as the starting point for the paper's MICE transfer.

In [ ]:
CROSS_MODEL = 'cross-encoder/ettin-reranker-68m-v1'
cross_tokenizer = AutoTokenizer.from_pretrained(CROSS_MODEL)
pair = cross_tokenizer(queries[0]['text'], corpus[0]['text'])
print('One jointly encoded pair (first 40 tokens):')
print(cross_tokenizer.convert_ids_to_tokens(pair['input_ids'])[:40])

In [ ]:
cross_encoder = CrossEncoder(CROSS_MODEL, device=device)
cross_pairs = [(query, document) for query in query_texts for document in documents]
_ = cross_encoder.predict(cross_pairs[:1], show_progress_bar=False)  # warm-up
start = time.perf_counter()
cross_scores = np.asarray(
    cross_encoder.predict(cross_pairs, batch_size=16, show_progress_bar=False)
).reshape(len(queries), len(corpus))
cross_seconds = time.perf_counter() - start

print('score matrix shape:', cross_scores.shape)
print('scoring seconds:', round(cross_seconds, 4))
print('top document per query:', [doc_ids[i] for i in cross_scores.argmax(axis=1)])

## 3. ColBERT: precomputed token vectors plus MaxSim

ColBERT independently creates a matrix for each document and query. If a query has $m$ token vectors and a document has $n$, their similarity matrix has shape $m × n$. For every query token, MaxSim keeps the largest similarity to any document token, then sums those maxima:

`score(q, d) = sum_i max_j similarity(q_i, d_j)`

The document matrices do not depend on the query, so they can be computed once and indexed.

In [ ]:
COLBERT_MODEL = 'lightonai/colbertv2.0'
colbert = MultiVectorEncoder(COLBERT_MODEL, device=device)

start = time.perf_counter()
doc_embeddings = colbert.encode_document(documents)
colbert_index_seconds = time.perf_counter() - start

start = time.perf_counter()
colbert_rows = []
query_embeddings = []
for query in query_texts:
    q_emb = colbert.encode_query(query)
    query_embeddings.append(q_emb)
    row = colbert.similarity(q_emb, doc_embeddings).squeeze(0).detach().cpu().numpy()
    colbert_rows.append(row)
colbert_query_seconds = time.perf_counter() - start
colbert_scores = np.vstack(colbert_rows)

print('first query matrix:', tuple(query_embeddings[0].shape))
print('first document matrix:', tuple(doc_embeddings[0].shape))
print('one-time document encoding seconds:', round(colbert_index_seconds, 4))
print('query encoding + MaxSim seconds:', round(colbert_query_seconds, 4))
print('top document per query:', [doc_ids[i] for i in colbert_scores.argmax(axis=1)])

### Inspect MaxSim instead of treating it as a black box

Below, rows are query-token vectors and columns are document-token vectors. The highlighted operation is conceptually: choose one maximum per row and sum. Query padding and special vectors are part of the trained ColBERT representation, so we inspect vector positions rather than pretending every row maps cleanly to a visible word.

In [ ]:
q_emb = query_embeddings[0]
d_emb = doc_embeddings[0]
similarity_matrix = q_emb @ d_emb.T
best_value, best_document_position = similarity_matrix.max(dim=1)
manual_maxsim = best_value.sum().item()
library_maxsim = float(colbert_scores[0, 0])

inspection = pd.DataFrame({
    'query_vector_position': range(min(12, len(best_value))),
    'best_document_position': best_document_position[:12].cpu().numpy(),
    'max_similarity': best_value[:12].detach().cpu().numpy(),
})
display(inspection)
print('manual MaxSim:', round(manual_maxsim, 5))
print('library score:', round(library_maxsim, 5))
print('close:', np.isclose(manual_maxsim, library_maxsim, atol=1e-4))

## 4. MICE-P: cached lower document layers plus learned interaction layers

The released MICE-P model exposes a cross-encoder-like `predict(query, document)` interface, but its internals are different. The checkpoint has 10 contextualization layers, 4 interaction layers, and a 128-dimensional document bottleneck.

This checkpoint contains custom model code, so `trust_remote_code=True` executes the implementation published inside the model repository. In a security-sensitive setting, inspect `modernbert_hf.py` before enabling it.

In [ ]:
MICE_MODEL = 'xpmir/Mice-P-61m'
mice = CrossEncoder(MICE_MODEL, trust_remote_code=True, device=device)
cfg = mice.model.config
print({
    'hidden_size': cfg.hidden_size,
    'contextualization_layers': cfg.n_contextualization_layers,
    'interaction_layers': cfg.n_interaction_layers,
    'document_bottleneck_dim': cfg.doc_bottleneck_dim,
    'uses_cross_attention': cfg.use_cross_attn,
})

In [ ]:
mice_pairs = [(query, document) for query in query_texts for document in documents]
_ = mice.predict(mice_pairs[:1], show_progress_bar=False)  # warm-up
start = time.perf_counter()
mice_scores = np.asarray(
    mice.predict(mice_pairs, batch_size=16, show_progress_bar=False)
).reshape(len(queries), len(corpus))
mice_seconds = time.perf_counter() - start

print('score matrix shape:', mice_scores.shape)
print('pair scoring seconds:', round(mice_seconds, 4))
print('top document per query:', [doc_ids[i] for i in mice_scores.argmax(axis=1)])

### A crucial limitation of this small demo

The convenient `predict` API scores text pairs and does not expose the paper's full offline FastPLAID store. The paper's end-to-end MICE-P system first uses SPLADEv3 to retrieve candidates, then loads/decompresses their cached document states and applies the interaction layers. Here we score every toy document to isolate ranking behavior.

## 5. Compare rankings—not raw score magnitudes

Each model was trained with a different scoring function and scale. A ColBERT score of 25 is not inherently better than a cross-encoder score of 10. We compare document order and reciprocal rank instead.

In [ ]:
all_scores = {
    'Cross-encoder': cross_scores,
    'ColBERT': colbert_scores,
    'MICE-P': mice_scores,
}
ranking = pd.DataFrame(
    sum((ranking_rows(name, scores) for name, scores in all_scores.items()), [])
)
display(ranking[ranking['rank'] <= 3].sort_values(['query_id', 'model', 'rank']))

summary = []
for name, scores in all_scores.items():
    rr = reciprocal_ranks(scores)
    summary.append({
        'model': name,
        'mean_reciprocal_rank': np.mean(rr),
        'top1_correct': sum(value == 1.0 for value in rr),
    })
display(pd.DataFrame(summary).sort_values('mean_reciprocal_rank', ascending=False))

In [ ]:
timing = pd.DataFrame([
    {'component': 'Cross-encoder: score all pairs', 'seconds': cross_seconds},
    {'component': 'ColBERT: encode documents once', 'seconds': colbert_index_seconds},
    {'component': 'ColBERT: encode queries + MaxSim', 'seconds': colbert_query_seconds},
    {'component': 'MICE-P predict API: score all pairs', 'seconds': mice_seconds},
])
display(timing)
print('Treat these as a local sanity check, not a speed benchmark: model loading is excluded, batching differs, and the corpus is tiny.')

## 6. Experiment with your own query

Change `custom_query` and inspect disagreements. A disagreement is often more educational than agreement: look for lexical traps, negation, entity ambiguity, or a need to combine several parts of a document.

In [ ]:
custom_query = 'Which document is about Jaguar the car manufacturer rather than the animal?'

custom_cross = np.asarray(cross_encoder.predict(
    [(custom_query, document) for document in documents], show_progress_bar=False
))
custom_q_emb = colbert.encode_query(custom_query)
custom_colbert = colbert.similarity(custom_q_emb, doc_embeddings).squeeze(0).detach().cpu().numpy()
custom_mice = np.asarray(mice.predict(
    [(custom_query, document) for document in documents], show_progress_bar=False
))

for name, scores in {
    'Cross-encoder': custom_cross,
    'ColBERT': custom_colbert,
    'MICE-P': custom_mice,
}.items():
    order = np.argsort(-scores)[:3]
    print(f'\n{name}')
    for rank, i in enumerate(order, 1):
        print(rank, doc_ids[i], round(float(scores[i]), 4), documents[i])

## 7. What to remember

- Choose **ColBERT** when you want an end-to-end neural retriever whose document token vectors can be indexed and whose query-time interaction is a simple, inspectable MaxSim operation.
- Choose a **cross-encoder** when reranking a manageable candidate set and maximum interaction flexibility matters more than repeated document computation.
- Think of **MICE-P** as a hybrid: cache substantial document computation, then retain learned attention-based interaction at scoring time. In the paper's current pipeline, this still needs a separate SPLADE first-stage retriever and a FastPLAID document store.
- A toy ranking is not evidence that one architecture is universally better. A fair experiment needs matched training, the same candidate sets, standard retrieval datasets, nDCG/Recall/MRR, index size, throughput, latency, and hardware controls—the kind of controlled setup the paper studies.